In [ ]:
# ─── Cell 0: Definitive Installation with Gradio Upgrade ──────────────────────
!pip install --upgrade \
  "langchain==0.1.16" \
  "langchain-core==0.1.45" \
  "langchain-openai==0.1.3" \
  "openai>=1.0.0" \
  "gradio>=4.0.0" \
  "chromadb<0.5.0" \
  "sentence-transformers<3.0.0" \
  "pypdf<4.0.0" \
  "tiktoken" \
  "transformers<5.0.0" \
  "python-dotenv<1.1.0" \
  "pdf2image<1.17.0" \
  "pillow<11.0.0"

# This part is still necessary for pdf2image to function correctly
!apt-get update -qq
!apt-get install -y --no-install-recommends poppler-utils

In [ ]:
# ─── Cell 1: Imports & ENV loading (Corrected for Pinned Versions) ───────────
import os, json, io, base64
from dotenv import load_dotenv
from typing import List
from PIL import Image
from pdf2image import convert_from_path
import gradio as gr

# 1) Load your .env (make sure it has OPENAI_API_KEY)
load_dotenv("template.env")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")


# 2) LangChain and OpenAI imports (updated for modern versions)
from openai import OpenAI as OpenAIClient
from langchain_openai import ChatOpenAI, OpenAI
from langchain_core.tools import tool
# FINAL CORRECTION: With the pinned versions, both key agent components are
# available directly from the top-level `langchain.agents` module.
from langchain.agents import AgentExecutor, create_openai_tools_agent
from langchain_core.messages import SystemMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

# 3) PDF parsing
from pypdf import PdfReader

In [ ]:
# ─── Cell 4: `vision_style_analyzer` Tool (Final, High-Detail Correction) ─────
@tool("vision_style_analyzer")
def vision_style_analyzer(pdf_path: str) -> dict:
    """
    Analyze the first page of the resume using GPT-4 Vision.

    Returns a dict with:
      - style_score (int): design score 1 (poor) to 10 (excellent)
      - template_type (str): e.g. 'classic', 'modern', 'creative'
      - suggestions (List[str]): concrete layout/design improvements
    """
    # THE FIX #1: Increase DPI for a much clearer source image.
    # The model can't analyze details it can't see.
    pages = convert_from_path(pdf_path, dpi=250, first_page=1, last_page=1)
    img = pages[0].convert("RGB")

    #

    # 2. Down-sample, but only if absolutely necessary (less aggressive).
    w, h = img.size
    if w > 2048:
        img = img.resize((2048, int(2048 * h / w)), Image.LANCZOS)

    # 3. Compress to JPEG @quality=75 (higher quality)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=75)
    buf.seek(0)

    # 4. Embed as base64 data URI
    b64_img = base64.b64encode(buf.getvalue()).decode("ascii")
    image_url = f"data:image/jpeg;base64,{b64_img}"

    # 5. Call GPT-4 Vision using the modern OpenAI v1.x+ client
    client = OpenAIClient()

    # THE FIX #2: A much more demanding and specific prompt.
    prompt_text = (
      "You are a senior hiring manager at a top tech firm with a background in graphic design. "
      "Your critique must be brutally honest and focused on what will get a candidate noticed or rejected based on visual presentation alone. "
      "Analyze the attached résumé image and provide a detailed critique. "
      "Do not give generic advice like 'add more white space' unless the document is genuinely cramped; instead, point to specific sections that need it. "
      "Your response MUST be a single, raw JSON object with the following keys:\n"
      "  'style_score': An integer from 1-10 based on its immediate professional impact.\n"
      "  'template_type': One of 'classic', 'modern', or 'creative'.\n"
      "  'positive_points': An array of strings detailing what is visually effective (e.g., 'Good use of columns for readability').\n"
      "  'improvement_suggestions': An array of actionable, specific strings for visual improvement (e.g., 'The font size for section headers is inconsistent; make all headers 14pt').\n\n"
      "Provide ONLY the raw JSON object and nothing else."
    )

    messages = [
        {
            "role": "system",
            "content": "You are an expert design critic providing feedback as a JSON object."
        },
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt_text},
                {
                    "type": "image_url",
                    "image_url": {
                        "url": image_url,
                        # THE FIX #3: Send the high-detail version of the image.
                        # This is critical for analyzing fonts, alignment, and spacing.
                        "detail": "high"
                    }
                }
            ]
        }
    ]

    resp = client.chat.completions.create(
        model="gpt-4o",  # Using the more powerful model for better vision analysis
        messages=messages,
        temperature=0.5, # Giving it slightly more room for nuanced wording
        response_format={"type": "json_object"}
    )

    # 6. Parse and return
    return json.loads(resp.choices[0].message.content)

In [ ]:
# ─── Cell 5: Define System Message ─────────────────────────────────────────────

SYSTEM_MESSAGE = SystemMessage(
    content="""
You are the Résumé Enhancement Agent. You only “communicate” by calling one of these three functions:

  1. parse_resume(pdf_path: str, job_spec: str) → {text, job_spec}
     • Always call this first to extract raw text against the spec.

  2. enhance_resume(text: str, job_spec: str) → {enhanced}
     • Use this on the parsed text to rewrite/improve the résumé.

  3. vision_style_analyzer(pdf_path: str) → {style_score, template_type, suggestions}
     • Use this when the user asks for a style critique or template feedback.

If the user asks to critique the visual design or template of the résumé, call vision_style_analyzer.
If the user asks to enhance content, follow the chain: parse_resume → enhance_resume.
"""
)

In [ ]:
# ─── Cell 7: Gradio UI (Corrected File Upload) ───────────────────────────────

with gr.Blocks() as demo:
    gr.Markdown("## AI‐Powered Résumé Enricher + Style Critic")

    with gr.Row():
        # THE FIX: We remove the `file_types=["pdf"]` argument.
        # This tells Gradio to accept any file and lets our Python code do the real validation.
        resume_file = gr.File(label="Upload Résumé (PDF)")
        job_spec    = gr.Textbox(label="Job Specification",
                                 placeholder="e.g. Senior Software Engineer")

    btn_enhance = gr.Button("Enhance Résumé")
    btn_style   = gr.Button("Critique Style")
    output      = gr.Textbox(label="Result", lines=20)

    # Function to handle the agent invocation and extract the output
    def run_agent(query):
        response = agent_executor.invoke({"input": query})
        return response['output']

    # Launch enhancement flow
    btn_enhance.click(
        fn=lambda pdf, spec: run_agent(
            f"Please enhance the résumé at {pdf.name} for a {spec} position."
        ),
        inputs=[resume_file, job_spec],
        outputs=output,
    )

    # Launch style-critique flow
    btn_style.click(
        fn=lambda pdf: run_agent(
            f"Please critique the visual design and template of the résumé at {pdf.name}."
        ),
        inputs=[resume_file],
        outputs=output,
    )

    demo.queue().launch(share=True, debug=True)